# 9.5 · 激活函数 / Activation Functions

> **课程定位 / Where this fits**
> 第 5 课，**Part 9 · 深度学习基础**。
> Lesson 5, **Part 9 · Deep Learning Foundations**.
>
> 9.1 说过：没有非线性激活，再深的网络也等于一层。这一课系统比较各种激活函数——**sigmoid / tanh / ReLU / LeakyReLU / GELU / Swish**——它们的形状、**梯度**、优缺点。重点是搞懂深度学习史上的关键转折：**为什么 ReLU 取代了 sigmoid**（答案藏在"梯度"里）。
> 9.1 noted: without nonlinear activations, any depth collapses to one layer. This lesson compares activations — **sigmoid / tanh / ReLU / LeakyReLU / GELU / Swish** — their shapes, **gradients**, and trade-offs. The key is the pivotal turn in DL history: **why ReLU replaced sigmoid** (the answer is in the gradients).
>
> 💼 **实战/面试视角**："为什么用 ReLU / 梯度消失 / dead ReLU / 各激活区别" 是深度学习高频题。
> 💼 **Practical/interview angle:** "why ReLU / vanishing gradients / dead ReLU / activation differences" — common DL questions.

> 📐 **符号约定 / Notation**
> - $\sigma(z)$ —— 激活函数 / activation
> - $\sigma'(z)$ —— 它的导数(决定梯度怎么流)/ its derivative (governs gradient flow)

> 💡 **面试相关 / Interview-relevant**
> - "为什么 ReLU 取代了 sigmoid"（出镜率 ★★★★★，梯度消失）
> - "梯度消失问题是什么"（出镜率 ★★★★★）
> - "dead ReLU 是什么 / 怎么解决"（★★★★，LeakyReLU）
> - "sigmoid/tanh/ReLU/GELU 的区别"（★★★★）
> - "输出层该用什么激活"（★★★★）

---

## 学习目标 / Learning Objectives

1. 画出各激活函数及其**导数**的形状。
   Plot each activation and its **derivative**.
2. 理解 sigmoid/tanh 的**梯度消失**问题。
   Understand sigmoid/tanh's **vanishing gradient** problem.
3. 理解 ReLU 为何胜出，及 **dead ReLU** 与 LeakyReLU。
   Understand why ReLU won, plus **dead ReLU** and LeakyReLU.
4. 了解 GELU/Swish 这些现代激活。
   Know modern activations GELU/Swish.
5. 知道**输出层**该用什么激活（任务决定）。
   Know what activation the **output layer** needs (task-dependent).

## 目录 / TOC
1. [先建直觉：激活 + 它的导数 ⭐](#1)
2. [梯度消失：sigmoid 的死穴 ⭐](#2)
3. [ReLU + dead ReLU + LeakyReLU ⭐](#3)
4. [现代激活：GELU / Swish ⭐](#4)
5. [实测对比 + 输出层选择 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：激活 + 它的导数 ⭐ / Intuition: Activation & Its Derivative

判断一个激活函数好不好，**关键不是看它本身的形状，而是看它的导数 $\sigma'(z)$**——因为反向传播(9.2)时，梯度每经过一个激活就要**乘上它的导数**。如果导数很小（接近 0），梯度乘着乘着就**消失**了，深层网络学不动。
What makes an activation good isn't its own shape but **its derivative $\sigma'(z)$** — because in backprop (9.2), the gradient is **multiplied by this derivative** each time it passes an activation. If the derivative is small (near 0), the gradient **vanishes** after several multiplications, and deep nets can't learn.

先把六个常用激活和它们的导数画出来，重点看**导数的最大值和"饱和区"**。
Let's plot six common activations and their derivatives, watching the **derivative's maximum value and "saturation regions"**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

z = np.linspace(-6, 6, 400)
def sigmoid(z): return 1/(1+np.exp(-z))
def d_sigmoid(z): s = sigmoid(z); return s*(1-s)
def d_tanh(z): return 1 - np.tanh(z)**2
relu = lambda z: np.maximum(0, z); d_relu = lambda z: (z > 0).astype(float)
lrelu = lambda z: np.where(z>0, z, 0.01*z); d_lrelu = lambda z: np.where(z>0, 1.0, 0.01)
gelu = lambda z: z * sigmoid(1.702*z)                      # GELU 的近似 / approximate GELU
swish = lambda z: z * sigmoid(z)                           # Swish = z·sigmoid(z)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, f in [("sigmoid", sigmoid), ("tanh", np.tanh), ("ReLU", relu),
                ("LeakyReLU", lrelu), ("GELU", gelu), ("Swish", swish)]:
    axes[0].plot(z, f(z), label=name)
axes[0].axhline(0, color="gray", lw=0.5); axes[0].legend(fontsize=8); axes[0].set_title("激活函数 activations")
for name, df in [("sigmoid'", d_sigmoid), ("tanh'", d_tanh), ("ReLU'", d_relu), ("LeakyReLU'", d_lrelu)]:
    axes[1].plot(z, df(z), label=name)
axes[1].axhline(0, color="gray", lw=0.5); axes[1].legend(fontsize=8)
axes[1].set_title("导数 derivatives (决定梯度怎么流; 注意 sigmoid' 最大才 0.25)")
plt.tight_layout(); plt.show()
print("sigmoid' 最大值只有 0.25, 且两端趋于 0(饱和); ReLU' 在正区恒为 1 → 这是关键区别")


<a id="2"></a>
## 2. 梯度消失：sigmoid 的死穴 ⭐ / Vanishing Gradient: Sigmoid's Flaw

**梯度消失(vanishing gradient)** 是深度学习早期最大的障碍，也是面试必考。看 sigmoid 的导数：最大值只有 **0.25**，且当输入较大/较小时趋于 **0**（"饱和"）。
**Vanishing gradient** was the biggest early obstacle in DL and a must-know. Look at sigmoid's derivative: its max is only **0.25**, and it goes to **0** for large/small inputs ("saturation").

反向传播时，梯度要**连乘**每一层的激活导数。如果每层都 ≤0.25，那 $L$ 层之后梯度被乘以 $\le 0.25^L$——10 层就乘以约 $10^{-6}$，**梯度几乎归零**，靠近输入的层根本学不动。这就是为什么用 sigmoid 的深层网络极难训练。
In backprop the gradient is **multiplied** by each layer's activation derivative. If each is ≤0.25, after $L$ layers it's scaled by $\le 0.25^L$ — at 10 layers that's ~$10^{-6}$, the gradient **nearly vanishes**, and early layers can't learn. This is why deep sigmoid nets are nearly impossible to train.


In [ ]:
# 模拟梯度连乘: 经过 L 层后梯度被缩放多少 / how the gradient scales after L layers
print("反向传播中梯度连乘各层激活导数, 经过 L 层后的缩放(假设每层都在最有利点):")
print(f"{'层数 L':<8}{'sigmoid (×0.25)':>18}{'ReLU (×1.0)':>14}")
for L in [1, 5, 10, 20]:
    print(f"{L:<8}{0.25**L:>18.2e}{1.0**L:>14.2f}")
print("\nsigmoid: 即使最理想(每层×0.25), 10 层后梯度只剩 ~1e-6 → 梯度消失, 浅层学不动")
print("ReLU: 正区导数恒为 1, 连乘不衰减 → 梯度能传到很深 → 这是 ReLU 取代 sigmoid 的根本原因")


<a id="3"></a>
## 3. ReLU + dead ReLU + LeakyReLU ⭐ / ReLU, Dead ReLU, LeakyReLU

**ReLU** $=\max(0, z)$ 是现代深度学习的默认激活，原因：(1) **正区导数恒为 1**，梯度不衰减 → 解决梯度消失；(2) 计算极快（就一个比较）；(3) 带来稀疏激活。
**ReLU** $=\max(0, z)$ is the default in modern DL because: (1) its **derivative is 1 in the positive region**, so gradients don't shrink → solves vanishing gradients; (2) it's extremely fast (one comparison); (3) it gives sparse activations.

**但 ReLU 有个坑：dead ReLU（死亡 ReLU）**。当一个神经元的输入总是负的（如学习率太大把权重推到一个总输出负值的状态），它的输出恒为 0、**导数也恒为 0 → 梯度永远是 0 → 这个神经元再也学不动了，永久"死亡"**。
**But ReLU has a pitfall: dead ReLU.** If a neuron's input is always negative (e.g. a too-large LR pushed weights to always output negative), its output is always 0 and **its derivative is always 0 → the gradient is always 0 → the neuron can never learn again, permanently "dead"**.

**LeakyReLU** 解决它：负区给一个小斜率（如 0.01z）而不是 0，这样负区也有微小梯度，神经元不会彻底死掉。
**LeakyReLU** fixes this: a small slope (e.g. 0.01z) in the negative region instead of 0, so there's a tiny gradient there and neurons don't fully die.


In [ ]:
import torch
import torch.nn as nn

# 演示 dead ReLU: 一个总输出负值的神经元, 梯度恒为 0 / a dead ReLU neuron
z_neg = torch.tensor([-3.0, -1.0, -0.5], requires_grad=True)
out = torch.relu(z_neg).sum(); out.backward()
print(f"ReLU 在全负输入处的梯度: {z_neg.grad.tolist()} → 全 0! (神经元'死了', 永远学不动)")

z_neg2 = torch.tensor([-3.0, -1.0, -0.5], requires_grad=True)
out2 = nn.functional.leaky_relu(z_neg2, 0.01).sum(); out2.backward()
print(f"LeakyReLU 在全负输入处的梯度: {z_neg2.grad.tolist()} → 小但非0 → 神经元还能恢复")
print("\ndead ReLU: 输入恒负 → 输出恒0 → 梯度恒0 → 永久死亡; 常因学习率过大引发")
print("LeakyReLU(负区小斜率) / PReLU / ELU 都是为缓解 dead ReLU; 也可调小学习率")


<a id="4"></a>
## 4. 现代激活：GELU / Swish ⭐ / Modern: GELU / Swish

近年的网络（尤其 Transformer，Part 13）常用更平滑的激活：
Recent networks (especially Transformers, Part 13) often use smoother activations:
- **GELU（高斯误差线性单元）**：$z\cdot\Phi(z)$（$\Phi$ 是标准正态 CDF）。它像一个"平滑版 ReLU"——在 0 附近平滑过渡而非硬拐角，且允许小的负值通过。**BERT、GPT 等都用它**。
  **GELU:** $z\cdot\Phi(z)$ ($\Phi$ = standard normal CDF). A "smooth ReLU" — smooth transition near 0 instead of a hard corner, allowing small negative values through. **Used in BERT, GPT, etc.**
- **Swish（也叫 SiLU）**：$z\cdot\sigma(z)$。同样平滑、非单调，由神经架构搜索发现，常略优于 ReLU。
  **Swish (a.k.a. SiLU):** $z\cdot\sigma(z)$. Also smooth and non-monotonic, found by neural architecture search, often slightly beats ReLU.

它们都保留了 ReLU"正区不饱和"的优点（避免梯度消失），又通过平滑获得更好的优化性质。实务里：**默认 ReLU，Transformer/追性能时用 GELU/Swish**。
Both keep ReLU's "non-saturating positive region" (avoiding vanishing gradients) while gaining better optimization from smoothness. In practice: **default to ReLU; use GELU/Swish for Transformers/peak performance**.


In [ ]:
# 对比 ReLU / GELU / Swish 在 0 附近的平滑度 / smoothness near 0
z = np.linspace(-3, 3, 300)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z, np.maximum(0, z), label="ReLU (硬拐角)", lw=2)
ax.plot(z, z*sigmoid(1.702*z), label="GELU (平滑, BERT/GPT 用)", lw=2)
ax.plot(z, z*sigmoid(z), label="Swish/SiLU (平滑非单调)", lw=2)
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.legend(); ax.set_title("现代激活: GELU/Swish 是'平滑版 ReLU', 0 附近无硬拐角 + 允许小负值")
plt.tight_layout(); plt.show()
print("GELU/Swish 保留 ReLU 正区不饱和的优点(不梯度消失) + 平滑(更好优化)")
print("实务: 默认 ReLU; Transformer/追性能用 GELU/Swish")


<a id="5"></a>
## 5. 实测对比 + 输出层选择 + 小结 ⭐ / Empirical Comparison, Output Layer & Summary

在 Digits 上用**不同激活**训同一个网络，对比收敛——通常 ReLU 系明显快于/好于 sigmoid（尤其层数多时）。
We train the same net on Digits with **different activations** and compare convergence — ReLU-family usually clearly beats sigmoid, especially with more layers.

**输出层激活是另一回事，由任务决定**（面试常考）：
**The output-layer activation is a separate matter, decided by the task** (often asked):
- **回归** → **无激活**（直接线性输出）。
  Regression → **none** (linear output).
- **二分类** → **sigmoid**（输出一个 [0,1] 概率）。
  Binary classification → **sigmoid** (one probability in [0,1]).
- **多分类** → **softmax**（输出概率分布）。
  Multi-class → **softmax** (a probability distribution).

注意：sigmoid 在**隐藏层**因梯度消失被淘汰，但在**输出层**做二分类仍是标准选择——它们是两个独立问题。
Note: sigmoid is obsolete in **hidden layers** (vanishing gradient) but still standard in the **output layer** for binary classification — two separate matters.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits(); X = digits.data/16.0
X_tr, X_te, y_tr, y_te = train_test_split(X, digits.target, test_size=0.3, stratify=digits.target, random_state=0)
Xtr_t = torch.tensor(X_tr, dtype=torch.float32); ytr_t = torch.tensor(y_tr)
Xte_t = torch.tensor(X_te, dtype=torch.float32)

def make_net(act):     # 故意用较深网络(4 隐藏层)放大激活的差异 / a deeper net amplifies differences
    layers = []
    dims = [64, 64, 64, 64, 64]
    for i in range(len(dims)-1):
        layers += [nn.Linear(dims[i], dims[i+1]), act()]
    layers += [nn.Linear(dims[-1], 10)]
    return nn.Sequential(*layers)

print(f"{'隐藏层激活':<16}{'test 准确率':>12}")
for name, act in [("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU), ("GELU", nn.GELU)]:
    torch.manual_seed(0)
    net = make_net(act); opt = torch.optim.Adam(net.parameters(), lr=1e-2); ce = nn.CrossEntropyLoss()
    for _ in range(80):
        opt.zero_grad(); ce(net(Xtr_t), ytr_t).backward(); opt.step()
    acc = (net(Xte_t).argmax(1).numpy() == y_te).mean()
    print(f"{name:<16}{acc:>12.3f}")
print("\n深网络下 ReLU/GELU 明显优于 Sigmoid(后者梯度消失, 浅层学不动)")
print("隐藏层默认 ReLU; 输出层按任务: 回归→无激活, 二分类→sigmoid, 多分类→softmax")


```
判断激活看导数 σ'(z): 反向传播梯度连乘各层 σ' → σ' 小则梯度消失
sigmoid/tanh: 两端饱和, σ' 趋0(sigmoid' 最大才 0.25) → 深层梯度消失(早期 DL 最大障碍)
ReLU=max(0,z): 正区 σ'=1 不衰减 → 解决梯度消失 + 快 + 稀疏 → 现代默认
dead ReLU: 输入恒负→输出恒0→梯度恒0→永久死亡(常因学习率大); LeakyReLU(负区小斜率)缓解
GELU/Swish: 平滑版 ReLU, Transformer 标配; 保留不饱和优点+更好优化
输出层(任务决定): 回归→无, 二分类→sigmoid, 多分类→softmax (与隐藏层用什么无关)
```

### 💡 面试速查 / Interview cheat-sheet
1. **ReLU 取代 sigmoid 因梯度消失**: sigmoid' 最大 0.25 连乘归零; ReLU 正区导数恒 1。
   ReLU replaced sigmoid due to vanishing gradients: sigmoid' ≤0.25 vanishes; ReLU's positive derivative is 1.
2. **梯度消失**: 反向梯度连乘小导数 → 浅层学不动。
   Vanishing gradient: backprop multiplies small derivatives → early layers can't learn.
3. **dead ReLU**(输入恒负→梯度恒0永久死) → LeakyReLU/调小学习率。
   Dead ReLU (always-negative input → zero gradient forever) → LeakyReLU / smaller LR.
4. **GELU/Swish** 是平滑 ReLU, Transformer 标配。
   GELU/Swish are smooth ReLUs, standard in Transformers.
5. **输出层激活按任务**: 回归无 / 二分类 sigmoid / 多分类 softmax。
   Output activation by task: none / sigmoid / softmax.

### 下一节 / Next
**9.6 损失函数**——激活决定网络怎么算, 损失决定它优化什么目标。MSE/交叉熵/Focal/对比/三元组损失各对应什么任务。
**9.6 Loss Functions** — activations decide how the net computes; losses decide what it optimizes. MSE/cross-entropy/focal/contrastive/triplet and their tasks.
